<a href="https://colab.research.google.com/github/AbdulH08/Percobaan-2/blob/main/deployvggstreamlit_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import gdown

url = 'https://drive.google.com/uc?id=17tPbN2_7ZyBriedo7gHEB2KEYPE90rRs&export=download'  # Ganti dengan file ID Google Drive
output = 'modelVGG16ep24.h5'
gdown.download(url, output, quiet=False)


Downloading...
From (original): https://drive.google.com/uc?id=17tPbN2_7ZyBriedo7gHEB2KEYPE90rRs&export=download
From (redirected): https://drive.google.com/uc?id=17tPbN2_7ZyBriedo7gHEB2KEYPE90rRs&export=download&confirm=t&uuid=ee4afdd2-73bd-405a-aed7-887681bd5d7b
To: /content/modelVGG16ep24.h5
100%|██████████| 213M/213M [00:03<00:00, 54.9MB/s]


'modelVGG16ep24.h5'

In [ ]:
!pip install tensorflow==2.13.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 524.1/524.1 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 440.8/440.8 kB 37.3 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.12.2
    Uninstalling typing_extensions-4.12.2:
      Successfully uninstalled typing_extensions-4.12.2
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: keras
    Found existing installation: keras 3.4.1
    Uninstalling keras-3.4.1:
      Successfully uninstalled keras-3.4.1
  Attempting uninstall: gast
    Found existing installation: gast 0.6.0
    Uninstalling gast-0.6.0:
     

In [ ]:
# Langkah 1: Instal Streamlit dan Ngrok
!pip install streamlit
!pip install pyngrok

# Langkah 2: Tambahkan authtoken ngrok yang Anda dapatkan
authtoken = "2hv1ooNcvUTecrns1LQZPNuHjEc_5HjEnceCo4vcaHKtW6rXD"

from pyngrok import ngrok

# Mengautentikasi ngrok
ngrok.set_auth_token(authtoken)
# Contoh model CNN yang disimpan dengan Keras
from tensorflow.keras.models import load_model

# Memuat model dari file
model = load_model('/content/drive/My Drive/modelVGG16ep24.h5')

!pip install streamlit
import streamlit as st
import tensorflow as tf
from tensorflow.keras.models import load_model
from PIL import Image
import numpy as np

# Memuat model CNN
model = load_model('modelVGG16ep24.h5')



In [ ]:
%%writefile app.py
import streamlit as st
import tensorflow as tf
import numpy as np
from PIL import Image

# Fungsi untuk memuat model (gantilah 'model_path' dengan path model Anda)
@st.cache(allow_output_mutation=True)
def load_model():
    model = tf.keras.models.load_model('/content/drive/My Drive/modelVGG16ep24.h5')  # Ganti dengan path model Anda
    return model

# Memuat model
model = load_model()

# Fungsi untuk memprediksi gambar menggunakan model CNN
def predict(image, model):
    # Mengubah ukuran gambar sesuai dengan input model
    image = image.resize((224, 224))  # Sesuaikan dengan ukuran input model Anda
    # Mengubah gambar menjadi array numpy dan menormalkan
    image = np.array(image) / 255.0
    # Menambahkan dimensi batch
    image = np.expand_dims(image, axis=0)
    # Membuat prediksi
    predictions = model.predict(image)
    return predictions

# Judul aplikasi
st.title("Alat Deteksi Penyakit Mata dengan CNN")

# Mengunggah gambar
uploaded_file = st.file_uploader("Unggah gambar untuk deteksi", type=["jpg", "png", "jpeg"])

if uploaded_file is not None:
    # Menampilkan gambar yang diunggah
    image = Image.open(uploaded_file)
    st.image(image, caption='Gambar yang diunggah', use_column_width=True)

    # Membuat prediksi jika model sudah dimuat
    if model is not None:
        predictions = predict(image, model)

        # Menampilkan hasil prediksi
        st.write("Hasil Prediksi:", predictions)

        # Mendapatkan indeks kelas yang diprediksi
        predicted_class_index = np.argmax(predictions[0])

        # Menampilkan hasil prediksi dengan label
        if predicted_class_index == 0:
            st.write('Kelas Prediksi: armd')
        elif predicted_class_index == 1:
            st.write('Kelas Prediksi: cataract')
        elif predicted_class_index == 2:
            st.write('Kelas Prediksi: diabetic_retinopathy')
        elif predicted_class_index == 3:
            st.write('Kelas Prediksi: glaucoma')
        elif predicted_class_index == 4:
            st.write('Kelas Prediksi: normal')
        else:
            st.write('Kelas tidak dikenali')
    else:
        st.write("Mohon tunggu, model sedang dimuat...")

# Menjalankan aplikasi Streamlit
if __name__ == '__main__':
    st.write("Silakan unggah gambar untuk memulai deteksi.")


Writing app.py


In [ ]:
import os
from pyngrok import ngrok

def run_streamlit():
    # Membuka tunnel ngrok dengan parameter bind_tls
    public_url = ngrok.connect(addr=8501)
    print(f"Streamlit app is live at: {public_url}")

    # Menjalankan Streamlit
    os.system('streamlit run app.py')

# Langkah 5: Jalankan Streamlit
run_streamlit()

Streamlit app is live at: NgrokTunnel: "https://dbdf-35-198-222-10.ngrok-free.app" -> "http://localhost:8501"
